# 01 — Uji Anonymization (PII Redaction)
**SuaraLens** | Pengujian redaksi PII menggunakan Microsoft Presidio + custom recognizer Indonesia.


## Setup

Pastikan library Presidio sudah terinstall sebelum menjalankan notebook ini.


In [ ]:
# Install jika belum ada
import subprocess, sys
pkgs = ['presidio-analyzer', 'presidio-anonymizer']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Download spaCy model kecil jika belum ada
try:
    import spacy
    spacy.load('en_core_web_sm')
except OSError:
    print('Downloading spaCy model...')
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

print('Setup selesai.')


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import re
import pandas as pd
from pathlib import Path
from modules.anonymization import anonymize_text, detect_pii_regex

DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/anonymization_results.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset dimuat: {len(df):,} baris')


## 1. Ambil Sampel Teks yang Mengandung PII

In [ ]:
# Filter teks yang likely mengandung PII
mask_nim   = df['teks_aduan'].str.contains(r'\b332\d{7}\b', regex=True, na=False)
mask_hp    = df['teks_aduan'].str.contains(r'0[89]\d', regex=True, na=False)
mask_email = df['teks_aduan'].str.contains('@', na=False)
mask_nama  = df['teks_aduan'].str.contains(r'(?i)\bsaya\s+[A-Z][a-z]+', regex=True, na=False)

pii_mask = mask_nim | mask_hp | mask_email | mask_nama
df_pii   = df[pii_mask].sample(min(50, pii_mask.sum()), random_state=42).reset_index(drop=True)

print(f'Teks dengan indikasi PII: {pii_mask.sum():,} dari {len(df):,}')
print(f'Sampel diambil: {len(df_pii)} baris')


## 2. Jalankan Anonymization pada 50 Sampel

In [ ]:
results = []
for _, row in df_pii.iterrows():
    redacted, entities = anonymize_text(row['teks_aduan'])
    results.append({
        'id_aduan':          row['id_aduan'],
        'teks_original':     row['teks_aduan'],
        'teks_redacted':     redacted,
        'entities_detected': entities,
        'n_entities':        len(entities),
    })

df_results = pd.DataFrame(results)
print(f'Anonymization selesai. Total entitas terdeteksi: {df_results["n_entities"].sum()}')


## 3. Tabel Before/After (10 Contoh)

In [ ]:
sample10 = df_results.head(10)[['id_aduan', 'teks_original', 'teks_redacted', 'n_entities']]

pd.set_option('display.max_colwidth', 80)
display(sample10)


## 4. Perhitungan Recall Sederhana

In [ ]:
# Recall: berapa % PII regex yang berhasil ditangkap Presidio+custom
tp_nim, tp_hp, tp_email = 0, 0, 0
fn_nim, fn_hp, fn_email = 0, 0, 0

for r in results:
    orig     = r['teks_original']
    entities = [e['type'] for e in r['entities_detected']]
    regex_pii = detect_pii_regex(orig)

    if regex_pii['nim']:
        if 'NIM' in entities:
            tp_nim += len(regex_pii['nim'])
        else:
            fn_nim += len(regex_pii['nim'])

    if regex_pii['phone']:
        if 'PHONE_NUMBER' in entities:
            tp_hp += len(regex_pii['phone'])
        else:
            fn_hp += len(regex_pii['phone'])

    if regex_pii['email']:
        if 'EMAIL_ADDRESS' in entities:
            tp_email += len(regex_pii['email'])
        else:
            fn_email += len(regex_pii['email'])

def recall(tp, fn):
    return round(tp / (tp + fn) * 100, 1) if (tp + fn) > 0 else None

recall_report = {
    'NIM':   {'true_positive': tp_nim,   'false_negative': fn_nim,   'recall_pct': recall(tp_nim,   fn_nim)},
    'Phone': {'true_positive': tp_hp,    'false_negative': fn_hp,    'recall_pct': recall(tp_hp,    fn_hp)},
    'Email': {'true_positive': tp_email, 'false_negative': fn_email, 'recall_pct': recall(tp_email, fn_email)},
}

print('=== Recall Presidio+Custom Recognizer ===')
for entity, stats in recall_report.items():
    r = stats['recall_pct']
    print(f"  {entity}: TP={stats['true_positive']}, FN={stats['false_negative']}, Recall={r}%")


## 5. Simpan Hasil ke JSON

In [ ]:
output = {
    'generated_at':     pd.Timestamp.now().isoformat(),
    'total_sampled':    len(results),
    'recall_report':    recall_report,
    'sample_results':   results[:20],  # simpan 20 sampel untuk referensi dashboard
    'limitations': [
        'PERSON recognizer menggunakan NER spaCy yang dilatih corpus Inggris.',
        'Recall untuk nama Indonesia (terutama nama pendek/umum) kemungkinan rendah.',
        'Perlu human review sebagai lapis kedua sesuai desain human-in-the-loop.',
        'Custom regex NIM PENS: pola 332XXXXXXX (10 digit). Perlu update jika format berubah.',
    ]
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Hasil disimpan ke: {OUTPUT_PATH}')


## Catatan Limitasi

1. **PERSON recognizer**: spaCy `en_core_web_sm` dilatih corpus Inggris → recall nama Indonesia kemungkinan rendah
2. **NIM regex**: pola `332XXXXXX` (10 digit) spesifik PENS — perlu update jika format berubah
3. **Nama pendek/umum** (mis. "Budi", "Sari") mungkin tidak tertangkap NER
4. **Human-in-the-loop wajib**: anonymization otomatis hanya lapis pertama, reviewer manusia tetap diperlukan sebelum data dipublikasi
